# Run Ollama in Colab
---

[![5aharsh/collama](https://raw.githubusercontent.com/5aharsh/collama/main/assets/banner.png)](https://github.com/5aharsh/collama)

This is an example notebook which demonstrates how to run Ollama inside a Colab instance. With this you can run pretty much any small to medium sized models offerred by Ollama for free.

For the list of available models check [models being offerred by Ollama](https://ollama.com/library).


## Before you proceed
---

Since by default the runtime type of Colab instance is CPU based, in order to use LLM models make sure to change your runtime type to T4 GPU (or better if you're a paid Colab user). This can be done by going to **Runtime > Change runtime type**.

While running your script be mindful of the resources you're using. This can be tracked at **Runtime > View resources**.

## Running the notebook
---

After configuring the runtime just run it with **Runtime > Run all**. And you can start tinkering around. This example uses [Llama 3.2](https://ollama.com/library/llama3.2) to generate a response from a prompted question using [LangChain Ollama Integration](https://python.langchain.com/docs/integrations/chat/ollama/).

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [1]:
!sudo apt update
!sudo apt install -y pciutils
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,985 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu 

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [3]:
!ollama pull qwen3

## And that's it!
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.

If you have a use-case that can help out others feel free to add your notebook to [Collama](https://github.com/5aharsh/collama/fork)

In [4]:
!pip install langchain-ollama

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="qwen3")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What's the length of hypotenuse in a right angled triangle"})))

The length of the hypotenuse in a right-angled triangle can be determined using the **Pythagorean theorem**, which states:

$$
c = \sqrt{a^2 + b^2}
$$

**Step-by-Step Explanation:**

1. **Identify the sides:**  
   In a right-angled triangle, the two sides forming the right angle are called the **legs** (denoted as $a$ and $b$), and the side opposite the right angle is the **hypotenuse** (denoted as $c$).

2. **Apply the theorem:**  
   Square the lengths of the two legs ($a^2$ and $b^2$), add them together, and take the square root of the result. This gives the length of the hypotenuse.

3. **Example:**  
   If the legs are $a = 3$ and $b = 4$:  
   $$
   c = \sqrt{3^2 + 4^2} = \sqrt{9 + 16} = \sqrt{25} = 5
   $$

**Final Answer:**  
The hypotenuse length is $ \boxed{\sqrt{a^2 + b^2}} $, where $a$ and $b$ are the lengths of the other two sides.

In [6]:
import ollama
import json, re

def generate_bio_ollama(prompt):
    # Ollama handles the chat template and generation for you
    response = ollama.chat(
        model='qwen3',
        messages=[{'role': 'user', 'content': prompt + " /no_think"}],
        options={
            'temperature': 0,      # Equivalent to do_sample=False
            'num_predict': 2048,   # Equivalent to max_new_tokens
        }
    )

    # Extract just the text content
    content = response['message']['content']

    # Clean out any leftover reasoning tags if present
    return re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

In [7]:
!pip install wikipedia-api

import wikipediaapi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 7.1 MB/s eta 0:00:00


In [8]:
# Getting wiki page
def get_wiki_context(entity_name, lang="en", sentences=5):
    wiki = wikipediaapi.Wikipedia(language=lang, user_agent="research-bot/1.0")
    page = wiki.page(entity_name)
    if not page.exists():
        print(f"[WARN] No Wikipedia page found for '{entity_name}' in lang='{lang}'")
        return None
    return ". ".join(page.summary.split(". ")[:sentences])

# RAG generate
def generate_bio_with_rag(prompt, entity_name, lang="en", **kwargs):
    context = get_wiki_context(entity_name, lang=lang)
    if context:
        augmented_prompt = f"Use the following factual context:\n{context}\n\n{prompt}"
    else:
        augmented_prompt = prompt

    print("=" * 60)
    print("AUGMENTED PROMPT:")
    print("=" * 60)
    print(augmented_prompt)
    print("=" * 60 + "\n")

    return generate_bio_ollama(augmented_prompt, **kwargs)

In [9]:
# English RAG generation
en_bio_rag = generate_bio_with_rag(
    "Write a biography of Xi Jinping.",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nENGLISH (RAG - English Wiki):\n", en_bio_rag)

AUGMENTED PROMPT:
Use the following factual context:
Xi Jinping (born 15 June 1953) is a Chinese statesman and politician who has served as the general secretary of the Chinese Communist Party (CCP) and chairman of the Party Central Military Commission (CMC) since 2012, and the president of China and chairman of the State Central Military Commission since 2013. Xi has been the leader of the fifth generation of Chinese leadership since 2012.
The elder son of Chinese Red Army veteran Xi Zhongxun's second marriage to Qi Xin, Xi was born in Beijing and was exiled to rural village of Liangjiahe, Yanchuan County, Shaanxi, as a teenager following his father's purge during the Cultural Revolution. He lived in a yaodong in Liangjiahe, where he joined the CCP after several failed attempts and worked as the local party secretary. After studying chemical engineering at Tsinghua University as a worker-peasant-soldier student, Xi rose through the ranks politically in China's coastal provinces. Xi wa

In [ ]:
# Bengali w/ English RAG
bn_bio_rag_en = generate_bio_with_rag(
    "শি জিনপিংয়ের জীবনী লিখুন।",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nBENGALI (RAG - English Wiki):\n", bn_bio_rag_en)

# Bengali w/ Native RAG
bn_bio_rag_native = generate_bio_with_rag(
    "শি জিনপিংয়ের জীবনী লিখুন।",
    entity_name="শি জিনপিং",
    lang="bn"
)
print("\nBENGALI (RAG - Bengali Wiki):\n", bn_bio_rag_native)

In [ ]:
# Hindi w/ English RAG
hi_bio_rag_en = generate_bio_with_rag(
    "शी जिनपिंग की जीवनी लिखें।",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nHINDI (RAG - English Wiki):\n", hi_bio_rag_en)

# Hindi w/ Native RAG
hi_bio_rag_native = generate_bio_with_rag(
    "शी जिनपिंग की जीवनी लिखें।",
    entity_name="शी जिनपिंग",
    lang="hi"
)
print("\nHINDI (RAG - Hindi Wiki):\n", hi_bio_rag_native)

In [ ]:
# Swahili w/ English RAG
sw_bio_rag_en = generate_bio_with_rag(
    "Andika wasifu wa Xi Jinping.",
    entity_name="Xi Jinping",
    lang="en"
)
print("\nSWAHILI (RAG - English Wiki):\n", sw_bio_rag_en)

# Swahili w/ Native RAG
sw_bio_rag_native = generate_bio_with_rag(
    "Andika wasifu wa Xi Jinping.",
    entity_name="Xi Jinping",
    lang="sw"
)
print("\nSWAHILI (RAG - Swahili Wiki):\n", sw_bio_rag_native)

In [ ]:
# English - no retrieval
en_bio = generate_bio_ollama("Write a biography of Xi Jinping.")
print("ENGLISH:\n", en_bio)

In [ ]:
# Bengali - no retrieval
bn_bio = generate_bio_ollama("শি জিনপিং-এর জীবনী লিখুন।")
# "Write a biography of Xi Jinping."
print("\nBENGALI:\n", bn_bio)

In [ ]:
# Swahili - no retrieval
sw_bio = generate_bio_ollama("Andika wasifu wa Xi Jinping.")
# "Write a biography of Xi Jinping."
print("\nSWAHILI:\n", sw_bio)

In [ ]:
# Hindi - no retrieval
hi_bio = generate_bio_ollama("शी जिनपिंग की जीवनी लिखें।")
# "Write a biography of Xi Jinping."
print("\nHINDI:\n", hi_bio)

In [ ]:
!ollama ps

NAME            ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
qwen3:latest    500a1f067a9f    6.0 GB    100% GPU     4096       29 seconds from now    
